# Graha instance segmentation inference
This notebook runs Graha/Lunar-FM Mask R-CNN instance segmentation on Pipeline-generated Lunar WAC datacubes. Input selection follows the same `DATA_DICT` format as `instance_ibm_train.ipynb`.

The datacube loader canonicalizes source WAC bands to VIS (5) followed by UV (2), appends static bands, and uses the training normalization statistics. Tiled detections are translated to full-image coordinates and deduplicated across overlaps.

# Setup

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
logging.getLogger('rasterio._env').setLevel(logging.ERROR)

import torch

In [ ]:
# Run from lfm/notebooks, or adjust this for your HPC checkout.
repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
if not (repo_root / 'lfm').exists():
    raise FileNotFoundError('Cannot find lfm/ directory. Run this notebook from lfm/notebooks or update repo_root.')
sys.path.insert(0, str(repo_root))

from lfm.all_models.all_tasks.data.normalization import load_terramind_pretraining_stats
from lfm.all_models.all_tasks.graha_inference import GrahaInstanceModel
from lfm.all_models.inst_seg import build_graha_notebook_configs
from lfm.all_models.inst_seg.data_cube_inference import (
    load_and_configure_input_data,
    plot_instance_inference_results,
    preprocess_datacubes,
    sliding_window_instance_inference,
)
from lfm.full_model.inst_seg import instance_graha_components
print('Successfully imported LFM and Graha instance modules')

# User configuration

- `INPUT_ROOT_DIR`: directory containing Pipeline WAC and static datacubes.
- `GRAHA_PRETRAIN_DIR`: Graha pretraining configuration and modality statistics.
- `GRAHA_LIGHTNING_CHECKPOINT`: fine-tuned Graha Mask R-CNN instance checkpoint.
- `OUTPUT_DIR`: destination for instance inference plots.
- `VERBOSE`: enables loading, preprocessing, and tiled-inference diagnostics.
- `SCORE_THRESHOLD`: minimum object confidence before tile merging.
- `MASK_THRESHOLD`: per-instance mask probability threshold.
- `NMS_IOU_THRESHOLD`: duplicate suppression threshold across overlapping tiles.
- `DATA_DICT`: training-style modality and band configuration. The checkpoint architecture must match the selected modalities.

Semantic shape-loss weight and padding parameters are intentionally absent; they are not part of the instance Mask R-CNN workflow.

In [ ]:
INPUT_ROOT_DIR = Path('/explore/nobackup/projects/lfm/model_inputs/inference/WAC_Processed_AOI')
GRAHA_PRETRAIN_DIR = Path('/explore/nobackup/projects/lfm/ibm_model_pretrain_dir_v2')
GRAHA_LIGHTNING_CHECKPOINT = Path('/explore/nobackup/projects/lfm/model_inference/checkpoints/inst_seg/graha/model.ckpt')
OUTPUT_DIR = Path('./outputs/instance_inference')
VERBOSE = False
SCORE_THRESHOLD = 0.5
MASK_THRESHOLD = 0.5
NMS_IOU_THRESHOLD = 0.5

DATA_DICT = {
    'dataset_name': 'wac_static_instance_inference',
    'data_dir': str(INPUT_ROOT_DIR),
    'dataset_modality': 'wac_static',
    'selected_modalities': ['vis', 'uv', 'static'],
    'band_filters': {
        'vis': [0, 1, 2, 3, 4],
        'uv': [0, 1],
        'static': list(range(63)),
    },
    'excluded_nodata_values': [
        -32768.0,
        -3.4028226550889045e38,
        -3.4028230607370965e38,
        -3.4028234663852886e38,
    ],
}

## Load and configure input data

In [ ]:
MODEL_NATIVE_SIZE = 256
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_data = load_and_configure_input_data(INPUT_ROOT_DIR, DATA_DICT, verbose=VERBOSE)
images_raw = input_data['images_raw']
nodata_masks = input_data['nodata_masks']
file_pairs = input_data['file_pairs']
BAND_FILTER = input_data['band_filter']
n_channels = input_data['n_channels']
GRAHA_BACKEND_MODALITIES = input_data['backend_modalities']
GRAHA_INPUT_MODE = input_data['input_mode']
NORMALIZATION_MODALITY = input_data['normalization_modality']
print(f'Using {device}; modalities={input_data["selected_modalities"]}; channels={n_channels}')

## Build the instance task and load its checkpoint

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
notebook_configs = build_graha_notebook_configs(
    output_dir=OUTPUT_DIR, data_root=INPUT_ROOT_DIR, base_output_dir=OUTPUT_DIR,
    graha_pretrain_dir=GRAHA_PRETRAIN_DIR,
    graha_lightning_checkpoint=GRAHA_LIGHTNING_CHECKPOINT, data_dict=DATA_DICT,
    normalization_modality=NORMALIZATION_MODALITY,
    graha_input_modality_mode=GRAHA_INPUT_MODE,
    graha_backend_modalities=GRAHA_BACKEND_MODALITIES, validate_paths=False,
)
graha_config = notebook_configs.graha_config
deps = notebook_configs.dependencies
means, stds = load_terramind_pretraining_stats(
    graha_config.modality_info,
    normalization_modality=NORMALIZATION_MODALITY,
    band_filter=BAND_FILTER,
)
task_cls = instance_graha_components.make_downstream_object_detection_task_class(
    deps['LunarObjectDetectionTask']
)
sample_batch = {'image': torch.zeros(1, n_channels, MODEL_NATIVE_SIZE, MODEL_NATIVE_SIZE)}
graha_task = instance_graha_components.create_task(graha_config, task_cls, sample_batch).to(device)
instance_graha_components.load_lightning_checkpoint_state(graha_task, GRAHA_LIGHTNING_CHECKPOINT)
graha_task.eval()
model = GrahaInstanceModel(graha_task).to(device).eval()
print('Successfully loaded Graha instance checkpoint')

# Inference
The final cell preprocesses the four selected datacubes, performs tiled instance inference, merges duplicate detections, and saves `graha_instance_inference_viz.png`.

In [ ]:
images_graha = preprocess_datacubes(images_raw, means=means, stds=stds, nodata_masks=nodata_masks, verbose=VERBOSE)
predictions = sliding_window_instance_inference(images_graha, model, device=device, target_size=MODEL_NATIVE_SIZE, n_channels=n_channels, score_threshold=SCORE_THRESHOLD, mask_threshold=MASK_THRESHOLD, nms_iou_threshold=NMS_IOU_THRESHOLD, nodata_masks=nodata_masks, verbose=VERBOSE)
fig = plot_instance_inference_results(images_raw, predictions, file_pairs, OUTPUT_DIR, n_channels, nodata_masks=nodata_masks, verbose=VERBOSE)